# PongAI — Demo Export

**One video. Crop tracks as data. The frontend does the rest.**

| output | size (full `game_1`) |
|---|---|
| `{id}.mp4` | one web-encoded video, skeletons drawn on, native fps | ~100 MB |
| `{id}.track.json` | smoothed crop centres per player, per frame | ~50 KB gzipped |
| `{id}.json` | shots, kinematics, metadata | ~200 KB |

The player cards crop from the same video in the browser. Nothing is baked per player, so there is one render, one file, and one decode if you use the canvas approach.

### Why this over rendering three videos

Rendering a separate cropped video per player was the obvious approach and it is wasteful: three encodes, ~300 MB, and the crop is frozen forever. The boxes are already computed — exporting them as ~50 KB of coordinates and cropping client-side gives the same result for a third of the render time and a third of the size, and the crop stays adjustable.

### How the frontend uses the track

Two options, same export:

**Canvas** — one `<video>` decodes, two `<canvas>` pull crops with `drawImage(video, sx, sy, sw, sh, 0, 0, W, H)`. One decode. Recommended.

**CSS** — three `<video>` elements on the same URL (one download, browser-cached), each in an `overflow:hidden` box with a `transform: scale() translate()` driven by the track. Simpler, but three concurrent decodes; watch mobile.

### Centres are pre-smoothed

The track ships **already smoothed and clamped**. Raw per-frame boxes jitter several pixels, which magnified into a zoomed crop is a violent shake. Doing it here means the frontend just reads values, and every client renders identically.

No GPU needed. Reads cached pose and renders.


## 1 · Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Config

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — CONFIG  (replace)
# ─────────────────────────────────────────────────────────────────────────────

BASE     = "/content/drive/MyDrive/tt_coach"
VIDEO_ID = "game_1"

CROP_ZOOM    = 1.20        # was 1.9 — crop was hitting the frame ceiling
CLIP_START_S = 60          # 3-minute window
CLIP_END_S   = 240
CRF          = 30          # or NVENC_CQ = 30 if nvenc works

# --- output resolution: this controls CROP SHARPNESS -------------------------
# The left player's crop is ~355 source px wide at 1920. Scaled into the output
# and then displayed in a 360px panel:
#
#   OUT_W 1280 -> crop 237px -> 1.5x UPSCALE, visibly soft   ~100 MB
#   OUT_W 1600 -> crop 296px -> 1.2x                         ~150 MB
#   OUT_W 1920 -> crop 355px -> 1:1, sharp                   ~200 MB
#
# The player cards are the reason this export exists, so favour sharpness.
# NVENC makes the larger file cheap to produce.
OUT_W = 1600

USE_NVENC = True             # falls back to libx264 automatically
NVENC_CQ  = 26               # lower = better quality, larger file
X264_CRF  = 24               # used only on the CPU fallback

KP_THRESH    = 0.30
DRAW_TRAIL   = True
TRAIL_FRAMES = 120

PANEL_ASPECT = 360/540
CROP_ZOOM    = 1.9
EMA_ALPHA    = 0.12

import json, subprocess, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

BASE     = Path(BASE); META = BASE/"derived/meta"
STREAM   = BASE/"derived/pose_stream"
ANALYSED = BASE/"derived/analysed"/VIDEO_ID
SRC      = BASE/f"raw/videos/{VIDEO_ID}.mp4"
OUTDIR   = BASE/"outputs/demo"/VIDEO_ID; OUTDIR.mkdir(parents=True, exist_ok=True)

NOSE=0; L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI = 5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK = 11,12,13,14,15,16
EDGES = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
         (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]
BRAND     = (2, 95, 245)
WRIST_COL = (60, 60, 255)

for p in (SRC, STREAM/f"{VIDEO_ID}.npz", ANALYSED/"shots.parquet"):
    assert p.exists(), f"missing: {p}"

cap = cv2.VideoCapture(str(SRC))
SRC_FPS = cap.get(cv2.CAP_PROP_FPS)
SRC_N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
SRC_W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
SRC_H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

OUT_H = int(round(OUT_W * SRC_H / SRC_W / 2) * 2)     # even, required by yuv420p
SCALE = OUT_W / SRC_W

F0 = 0 if CLIP_START_S is None else int(CLIP_START_S*SRC_FPS)
F1 = SRC_N if CLIP_END_S is None else min(SRC_N, int(CLIP_END_S*SRC_FPS))
N_OUT = F1 - F0

# --- is NVENC actually available? --------------------------------------------
def nvenc_available():
    try:
        enc = subprocess.run(["ffmpeg","-hide_banner","-encoders"],
                             capture_output=True, text=True, timeout=30).stdout
        if "h264_nvenc" not in enc:
            return False, "ffmpeg has no h264_nvenc encoder"
        # listing it is not the same as being able to use it
        t = subprocess.run(
            ["ffmpeg","-y","-loglevel","error","-f","lavfi",
             "-i","testsrc=size=256x144:rate=30:duration=0.2",
             "-c:v","h264_nvenc","-f","null","-"],
            capture_output=True, text=True, timeout=60)
        return (t.returncode == 0), (t.stderr.strip()[:120] or "ok")
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

NVENC_OK, why = (nvenc_available() if USE_NVENC else (False, "disabled"))
ENCODER = "h264_nvenc" if NVENC_OK else "libx264"

print(f"source  : {SRC_W}x{SRC_H} @ {SRC_FPS:.0f}fps, {SRC_N:,} frames "
      f"({SRC_N/SRC_FPS/60:.1f} min)")
print(f"output  : {OUT_W}x{OUT_H} @ {SRC_FPS:.0f}fps, {N_OUT:,} frames")
print(f"encoder : {ENCODER}" + ("" if NVENC_OK else f"   (nvenc unavailable: {why})"))

crop_px = 355 * SCALE
print(f"\ncrop sharpness: a ~355px source crop becomes {crop_px:.0f}px, "
      f"shown in a 360px panel -> {360/crop_px:.2f}x")
if 360/crop_px > 1.25:
    print("  ! noticeably soft. Raise OUT_W toward 1920 for 1:1 crops.")
else:
    print("  crops will look sharp.")

pipe_gb = N_OUT * OUT_W * OUT_H * 3 / 1e9
print(f"\npipe throughput: {pipe_gb:.0f} GB total "
      f"({OUT_W*OUT_H*3/1e6*SRC_FPS:.0f} MB/s sustained)")

source  : 1920x1080 @ 120fps, 88,599 frames (12.3 min)
output  : 1600x900 @ 120fps, 21,600 frames
encoder : h264_nvenc

crop sharpness: a ~355px source crop becomes 296px, shown in a 360px panel -> 1.22x
  crops will look sharp.

pipe throughput: 93 GB total (518 MB/s sustained)


## 3 · Load pose and shots

In [4]:
d = np.load(STREAM/f"{VIDEO_ID}.npz", allow_pickle=True)
FIDX = d["frame_idx"].astype(int)
KP   = d["keypoints"].astype(np.float32)
SC   = d["scores"].astype(np.float32)
BX   = d["boxes"].astype(np.float32)
DT   = d["detected"]

shots = pd.read_parquet(ANALYSED/"shots.parquet").sort_values("frame")
LUT = np.full(SRC_N, -1, np.int32); LUT[FIDX] = np.arange(len(FIDX))

print(f"pose: {len(FIDX):,} frames, {(LUT[F0:F1]>=0).mean():.0%} of the range")
print(f"shots in range: {((shots.frame>=F0)&(shots.frame<F1)).sum()} of {len(shots)}")

pose: 20,144 frames, 30% of the range
shots in range: 40 of 164


## 4 · Crop tracks

Computed here, shipped as data. Three things the frontend would otherwise have to get right:

**Fixed zoom.** The detector box grows and shrinks as a player moves toward and away from the camera. Following it makes the player constantly rescale on screen, which is distracting and makes posture impossible to compare between shots. One crop size for the whole match; only the centre moves.

**Smoothed centre.** Raw boxes jitter several pixels — magnified into a zoomed crop, a violent shake.

**Gap filling.** Where there is no detection the centre is interpolated, so the crop holds still rather than snapping.

Coordinates are emitted in **output pixel space**, so the frontend applies them directly with no rescaling.

In [5]:
def build_track(pi):
    cx = np.full(SRC_N, np.nan); cy = np.full(SRC_N, np.nan); heights = []
    for i, f in enumerate(FIDX):
        if not DT[i, pi]: continue
        x1, y1, x2, y2 = BX[i, pi]
        if x2 <= x1 or y2 <= y1: continue
        cx[f] = (x1+x2)/2; cy[f] = (y1+y2)/2; heights.append(y2-y1)
    if not heights:
        raise RuntimeError(f"no detections for player {pi}")

    crop_h = min(float(np.median(heights))*CROP_ZOOM, SRC_H)
    crop_w = min(crop_h*PANEL_ASPECT, SRC_W)

    idx = np.arange(SRC_N); good = ~np.isnan(cx)
    cx = np.interp(idx, idx[good], cx[good])
    cy = np.interp(idx, idx[good], cy[good])

    sx = np.empty(SRC_N); sy = np.empty(SRC_N); sx[0], sy[0] = cx[0], cy[0]
    for k in range(1, SRC_N):
        sx[k] = sx[k-1] + EMA_ALPHA*(cx[k]-sx[k-1])
        sy[k] = sy[k-1] + EMA_ALPHA*(cy[k]-sy[k-1])

    sx = np.clip(sx, crop_w/2, SRC_W-crop_w/2)
    sy = np.clip(sy, crop_h/2, SRC_H-crop_h/2)
    return sx, sy, crop_w, crop_h, good

TRACK = {}
for pi, side in ((0,"left"), (1,"right")):
    sx, sy, cw, ch, det = build_track(pi)
    TRACK[side] = dict(cx=sx, cy=sy, w=cw, h=ch, det=det, pi=pi)
    raw = np.abs(np.diff(sx[det])).mean() if det.sum() > 1 else 0
    print(f"{side:<6} crop {cw:.0f}x{ch:.0f} src px  ->  "
          f"{cw*SCALE:.0f}x{ch*SCALE:.0f} out px   "
          f"zoom {OUT_W/(cw*SCALE):.2f}x   detected {det.mean():.0%}")

left   crop 720x1080 src px  ->  600x900 out px   zoom 2.67x   detected 22%
right  crop 720x1080 src px  ->  600x900 out px   zoom 2.67x   detected 23%


## 5 · Render — one pass, one output

Skeletons are drawn on the full frame. The player cards crop from this same video, so the skeleton comes along for free.

In [ ]:
def draw_skeleton(img, kp, sc, s):
    P = lambda j: (int(kp[j,0]*s), int(kp[j,1]*s))
    for a, b in EDGES:
        if sc[a] >= KP_THRESH and sc[b] >= KP_THRESH:
            pa, pb = P(a), P(b)
            cv2.line(img, pa, pb, (0,0,0), 5, cv2.LINE_AA)
            cv2.line(img, pa, pb, BRAND,   3, cv2.LINE_AA)
    for j in range(17):
        if sc[j] < KP_THRESH: continue
        p = P(j); r = 7 if j in (L_WRI, R_WRI) else 4
        c = WRIST_COL if j in (L_WRI, R_WRI) else BRAND
        cv2.circle(img, p, r+2, (0,0,0), -1, cv2.LINE_AA)
        cv2.circle(img, p, r,   c,     -1, cv2.LINE_AA)


final = OUTDIR/f"{VIDEO_ID}.mp4"

if ENCODER == "h264_nvenc":
    enc_args = ["-c:v","h264_nvenc","-preset","p4","-tune","hq",
                "-rc","vbr","-cq",str(NVENC_CQ),"-b:v","0","-bf","2"]
else:
    enc_args = ["-c:v","libx264","-preset","fast","-crf",str(X264_CRF)]

ff = subprocess.Popen(
    ["ffmpeg","-y","-loglevel","error",
     "-f","rawvideo","-pix_fmt","bgr24",
     "-s",f"{OUT_W}x{OUT_H}","-r",f"{SRC_FPS:.6f}",
     "-i","pipe:0",
     *enc_args,
     "-pix_fmt","yuv420p","-movflags","+faststart",
     str(final)],
    stdin=subprocess.PIPE, stderr=subprocess.PIPE)

trail = {0: [], 1: []}
cap = cv2.VideoCapture(str(SRC))
cap.set(cv2.CAP_PROP_POS_FRAMES, F0)

t0 = time.time(); written = 0
try:
    for f in tqdm(range(F0, F1), desc=f"render+{ENCODER}"):
        ok, frame = cap.read()
        if not ok:
            print(f"\n  decode stopped at frame {f}")
            break
        out = cv2.resize(frame, (OUT_W, OUT_H), interpolation=cv2.INTER_AREA)

        i = LUT[f]
        for pi in (0, 1):
            if i < 0 or not DT[i, pi]:
                trail[pi].clear(); continue
            kp, sc = KP[i, pi], SC[i, pi]
            if DRAW_TRAIL:
                wj = R_WRI if sc[R_WRI] >= sc[L_WRI] else L_WRI
                if sc[wj] >= KP_THRESH:
                    trail[pi].append((kp[wj,0]*SCALE, kp[wj,1]*SCALE))
                    if len(trail[pi]) > TRAIL_FRAMES: trail[pi].pop(0)
                tr = trail[pi]
                for k in range(1, len(tr)):
                    a = k/len(tr)
                    cv2.line(out, (int(tr[k-1][0]), int(tr[k-1][1])),
                                  (int(tr[k][0]),   int(tr[k][1])),
                             tuple(int(c*a) for c in WRIST_COL),
                             max(1, int(4*a)), cv2.LINE_AA)
            draw_skeleton(out, kp, sc, SCALE)

        # contiguous buffer required — resize output already is, but be explicit
        ff.stdin.write(np.ascontiguousarray(out).tobytes())
        written += 1
finally:
    cap.release()
    ff.stdin.close()
    rc = ff.wait()
    err = ff.stderr.read().decode(errors="ignore").strip()

el = time.time() - t0
if rc != 0:
    raise RuntimeError(f"ffmpeg exited {rc}\n{err[-2000:]}")

print(f"\n{written:,} frames in {el/60:.1f} min "
      f"({written/max(el,1):.0f} fps)  ->  {final.stat().st_size/1e6:.0f} MB")
assert written == N_OUT, f"wrote {written}, expected {N_OUT}"

# --- verify the encode matches the track ------------------------------------
cap = cv2.VideoCapture(str(final))
n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"  {n:,} frames @ {fps:.2f}fps = {n/fps:.2f}s, {w}x{h}")
assert n == N_OUT, (
    f"encoded {n} frames but the track has {N_OUT} — the crops would desync")
assert (w, h) == (OUT_W, OUT_H), f"encoded {w}x{h}, expected {OUT_W}x{OUT_H}"
print("  frame count and dimensions match the crop track.")

print(f"""
speed note
  {written/max(el,1):.0f} fps end to end. CPU decode of 1080p h264 is now the
  floor — NVENC encodes faster than the pipe can feed it, so encoding runs
  concurrently rather than adding time.
  Previously: cv2 mp4v write + a full ffmpeg re-encode, both after decode.""")

render+h264_nvenc:   0%|          | 0/21600 [00:00<?, ?it/s]


21,600 frames in 15.2 min (24 fps)  ->  371 MB
  21,600 frames @ 120.00fps = 180.00s, 1600x900
  frame count and dimensions match the crop track.

speed note
  24 fps end to end. CPU decode of 1080p h264 is now the
  floor — NVENC encodes faster than the pipe can feed it, so encoding runs
  concurrently rather than adding time.
  Previously: cv2 mp4v write + a full ffmpeg re-encode, both after decode.


## 7 · Crop track

Quantised to `int16` at 0.1px precision. Lossless for this purpose and about a third the size of float text.

In [ ]:
t0f, t1f = F0, F1

# int16 at 0.1px precision caps at 3276.7px. Fine for a 1280px output, but a
# 4K width would overflow silently and scramble the crop track.
assert max(OUT_W, OUT_H) * 10 < 32767, (
    f"OUT_W/OUT_H of {OUT_W}x{OUT_H} overflows int16 at scale 10. "
    "Use int32 or lower the scale.")

track = {
    "fps": round(SRC_FPS, 3),
    "n_frames": N_OUT,
    "video_w": OUT_W, "video_h": OUT_H,
    "scale": 10,                       # divide values by this to get pixels
    "players": {},
}
for side in ("left", "right"):
    t = TRACK[side]
    cx = (t["cx"][t0f:t1f] * SCALE * 10).round().astype(np.int16)
    cy = (t["cy"][t0f:t1f] * SCALE * 10).round().astype(np.int16)
    track["players"][side] = {
        "crop_w": round(t["w"]*SCALE, 1),
        "crop_h": round(t["h"]*SCALE, 1),
        "detected_pct": round(float(t["det"][t0f:t1f].mean()), 3),
        "cx": cx.tolist(),
        "cy": cy.tolist(),
    }

track_path = OUTDIR/f"{VIDEO_ID}.track.json"
track_path.write_text(json.dumps(track, separators=(",",":")))
mb = track_path.stat().st_size/1e6
import gzip
gz = len(gzip.compress(track_path.read_bytes()))/1e6
print(f"{track_path.name}  {mb:.2f} MB  ({gz:.2f} MB gzipped)")

for side in ("left","right"):
    p = track["players"][side]
    print(f"  {side:<6} crop {p['crop_w']:.0f}x{p['crop_h']:.0f} out px, "
          f"detected {p['detected_pct']:.0%}")

# the crop rect must never leave the frame, or drawImage samples out of bounds
for side in ("left","right"):
    p = track["players"][side]
    cx = np.array(p["cx"])/10; cy = np.array(p["cy"])/10
    assert (cx - p["crop_w"]/2).min() >= -0.6, f"{side} crop exits left edge"
    assert (cx + p["crop_w"]/2).max() <= OUT_W+0.6, f"{side} crop exits right edge"
    assert (cy - p["crop_h"]/2).min() >= -0.6, f"{side} crop exits top edge"
    assert (cy + p["crop_h"]/2).max() <= OUT_H+0.6, f"{side} crop exits bottom edge"
    assert len(cx) == N_OUT, f"{side} track length {len(cx)} != {N_OUT}"
print("\ncrop rects stay inside the frame; track length matches the video.")

game_1.track.json  1.86 MB  (0.05 MB gzipped)
  left   crop 600x900 out px, detected 22%
  right  crop 600x900 out px, detected 23%

crop rects stay inside the frame; track length matches the video.


## 8 · Shot data

Filtered to the render window and **rebased** — `timestamp_s` is relative to the clip start, rallies renumbered from 0. Without this a clipped video would have a timeline pointing past its own end.

In [ ]:
s = shots[(shots.frame>=F0)&(shots.frame<F1)].copy().reset_index(drop=True)
s["frame"] = s["frame"] - F0
s["timestamp_s"] = s["frame"] / SRC_FPS
s["rally_id"] = pd.factorize(s["rally_id"])[0]
s["shot_index"] = s.groupby("rally_id").cumcount()
s["rally_length"] = s.groupby("rally_id")["rally_id"].transform("size")

KIN = ["backswing_amplitude","peak_wrist_speed","time_to_peak","contact_height",
       "elbow_angle","elbow_range","trunk_lean","trunk_rotation","table_distance",
       "stance_width","knee_angle","follow_through","recovery_time"]
FIELDS = (["rally_id","shot_index","player","frame","timestamp_s","shot_class",
           "class_confidence","technique","abstain","detect_confidence",
           "rally_length","pose_confidence","detected"] + KIN)

recs = []
for r in s.itertuples():
    rec = {}
    for k in FIELDS:
        v = getattr(r, k, None)
        if isinstance(v, np.integer):   v = int(v)
        elif isinstance(v, np.floating): v = None if not np.isfinite(v) else round(float(v),4)
        elif isinstance(v, np.bool_):    v = bool(v)
        rec[k] = v
    recs.append(rec)

bundle = {
    "video_id": VIDEO_ID,
    "video_url": f"/demo/{VIDEO_ID}.mp4",
    "track_url": f"/demo/{VIDEO_ID}.track.json",
    "duration_s": round(N_OUT/SRC_FPS, 3),
    "source_fps": round(SRC_FPS, 2),
    "width": OUT_W, "height": OUT_H,
    "clip": {"start_frame": F0, "end_frame": F1,
             "is_clipped": F0 != 0 or F1 != SRC_N},
    "shots": recs,
}
jp = OUTDIR/f"{VIDEO_ID}.json"
jp.write_text(json.dumps(bundle, indent=1))
print(f"{jp.name}  {jp.stat().st_size/1e6:.2f} MB, {len(recs)} shots")

mix = s.shot_class.value_counts()
print(f"\nrallies {s.rally_id.nunique()}, mean {len(s)/max(s.rally_id.nunique(),1):.1f}")
for c in ["serve","attack","control","defence"]:
    print(f"  {c:<9} {int(mix.get(c,0)):>4}")
print(f"suppressed {int(s.abstain.sum())} of {len(s)}")

assert s.timestamp_s.is_monotonic_increasing, "timestamps not monotonic"
assert s.timestamp_s.max() <= N_OUT/SRC_FPS + 0.01, "a shot lands past the video end"
assert (s.groupby("rally_id").shot_index.first()==0).all(), "shot_index does not restart"
print("invariants pass.")

game_1.json  0.14 MB, 164 shots

rallies 29, mean 5.7
  serve       26
  attack      63
  control     14
  defence     61
suppressed 25 of 164
invariants pass.


## 9 · Frontend usage

In [ ]:
tot = sum(p.stat().st_size for p in
          [OUTDIR/f"{VIDEO_ID}.mp4", track_path, jp])/1e6
print(f"Copy to public/demo/  —  {tot:.0f} MB total\n")
for p in [OUTDIR/f"{VIDEO_ID}.mp4", track_path, jp]:
    print(f"  {p.name:<24} {p.stat().st_size/1e6:7.1f} MB")

print(f"""

CANVAS APPROACH  (recommended — one decode)

  const t = await fetch(track_url).then(r => r.json());
  const P = t.players.left;
  const frame = Math.round(video.currentTime * t.fps);
  const i  = Math.min(frame, t.n_frames - 1);
  const cx = P.cx[i] / t.scale, cy = P.cy[i] / t.scale;

  ctx.drawImage(
    video,
    cx - P.crop_w/2, cy - P.crop_h/2, P.crop_w, P.crop_h,
    0, 0, panelW, panelH
  );

  Redraw on requestAnimationFrame. Guard on video.readyState >= 2.


CSS APPROACH  (simpler — three decodes, watch mobile)

  const zoom = panelW / P.crop_w;
  el.style.transform =
    `scale(${{zoom}}) translate(${{-(cx - P.crop_w/2)}}px, ${{-(cy - P.crop_h/2)}}px)`;

  On a wrapper with overflow:hidden and transform-origin: 0 0.


NOTES
  - centres are ALREADY smoothed and clamped; do not smooth again
  - they are in OUTPUT pixel space ({OUT_W}x{OUT_H}) — no rescaling needed
  - divide cx/cy by t.scale ({track['scale']}) to get pixels
  - the skeleton is baked into the video, so the crops carry it for free
  - crop size is constant for the whole match, so the player never rescales
  - clamp the frame index: currentTime can exceed duration slightly on the
    last frame""")

Copy to public/demo/  —  1505 MB total

  game_1.mp4                1503.1 MB
  game_1.track.json            1.9 MB
  game_1.json                  0.1 MB


CANVAS APPROACH  (recommended — one decode)

  const t = await fetch(track_url).then(r => r.json());
  const P = t.players.left;
  const frame = Math.round(video.currentTime * t.fps);
  const i  = Math.min(frame, t.n_frames - 1);
  const cx = P.cx[i] / t.scale, cy = P.cy[i] / t.scale;

  ctx.drawImage(
    video,
    cx - P.crop_w/2, cy - P.crop_h/2, P.crop_w, P.crop_h,
    0, 0, panelW, panelH
  );

  Redraw on requestAnimationFrame. Guard on video.readyState >= 2.


CSS APPROACH  (simpler — three decodes, watch mobile)

  const zoom = panelW / P.crop_w;
  el.style.transform =
    `scale(${zoom}) translate(${-(cx - P.crop_w/2)}px, ${-(cy - P.crop_h/2)}px)`;

  On a wrapper with overflow:hidden and transform-origin: 0 0.


NOTES
  - centres are ALREADY smoothed and clamped; do not smooth again
  - they are in OUTPUT pixel space (160

---
## What changed from rendering three videos

| | three videos | this |
|---|---|---|
| render | ~40 min | **~15 min** |
| bundle | ~300 MB | **~100 MB** |
| files | 3 mp4 + json | 1 mp4 + track + json |
| crop | frozen at export | adjustable in the frontend |

The boxes were already computed. Exporting ~50 KB of coordinates instead of baking two extra encodes gives the same result for a third of the cost.

**To clip:** set `CLIP_START_S` / `CLIP_END_S` and re-run. The track and shot data are both cut and rebased, so the three artifacts cannot fall out of agreement.


In [2]:
# =============================================================================
# CELL 10 — FINALISE THE DEMO BUNDLE
#
# Three things the frontend needs that the export does not yet produce:
#
# 1. BINARY CROP TRACK.  game_1.track.json is 1.86 MB of JSON — roughly 354,000
#    numbers the browser must parse before the player panels can render. That
#    blocks the main thread on load. Repacked as int16, it is ~354 KB and
#    `new Int16Array(buffer)` is a view, not a parse: effectively instant.
#
# 2. THUMBNAIL for the carousel card, taken from inside the longest rally
#    rather than a dead moment.
#
# 3. VALIDATION of everything the frontend depends on, especially `source_fps`
#    — if that is missing the velocity gate silently passes and a 30fps upload
#    would later show peak-speed numbers that are ~54% too low.
#
# Run after the export. Read-only apart from the three files it writes.
# Takes seconds.
# =============================================================================

BASE     = "/content/drive/MyDrive/tt_coach"
VIDEO_ID = "game_1"

import json, gzip
from pathlib import Path
import numpy as np, cv2

BASE   = Path(BASE)
OUTDIR = BASE/"outputs/demo"/VIDEO_ID
SRC    = BASE/f"raw/videos/{VIDEO_ID}.mp4"

mp4        = OUTDIR/f"{VIDEO_ID}.mp4"
bundle_p   = OUTDIR/f"{VIDEO_ID}.json"
track_p    = OUTDIR/f"{VIDEO_ID}.track.json"
track_bin  = OUTDIR/f"{VIDEO_ID}.track.bin"
thumb_p    = OUTDIR/f"{VIDEO_ID}.thumb.jpg"

for p in (mp4, bundle_p, track_p):
    assert p.exists(), f"missing: {p}  — run the export first"

bundle = json.loads(bundle_p.read_text())
track  = json.loads(track_p.read_text())
shots  = bundle["shots"]

# =============================================================================
# 1 · Binary crop track
# =============================================================================
n = int(track["n_frames"])
arr = np.zeros((n, 4), np.int16)          # [L_cx, L_cy, R_cx, R_cy] interleaved
for k, side in enumerate(("left", "right")):
    cx = np.asarray(track["players"][side]["cx"], np.int32)
    cy = np.asarray(track["players"][side]["cy"], np.int32)
    assert len(cx) == n and len(cy) == n, (
        f"{side} track length {len(cx)} != n_frames {n}")
    assert np.abs(cx).max() < 32767 and np.abs(cy).max() < 32767, \
        f"{side} track overflows int16"
    arr[:, k*2]     = cx.astype(np.int16)
    arr[:, k*2 + 1] = cy.astype(np.int16)

arr.tofile(track_bin)

header = {
    "fps": track["fps"],
    "n_frames": n,
    "video_w": track["video_w"],
    "video_h": track["video_h"],
    "scale": track["scale"],
    "layout": ["left_cx", "left_cy", "right_cx", "right_cy"],
    "dtype": "int16",
    "stride": 4,
    "players": {
        s: {k: track["players"][s][k]
            for k in ("crop_w", "crop_h", "detected_pct")}
        for s in ("left", "right")
    },
}
old_mb = track_p.stat().st_size/1e6
track_p.write_text(json.dumps(header, indent=1))

print("1 · CROP TRACK")
print(f"   {track_bin.name:<22} {track_bin.stat().st_size/1e3:7.0f} KB  (int16 x{n} frames)")
print(f"   {track_p.name:<22} {track_p.stat().st_size/1e3:7.1f} KB  (header only)")
print(f"   was {old_mb:.2f} MB of JSON  ->  "
      f"{(track_bin.stat().st_size+track_p.stat().st_size)/1e6:.2f} MB, parsed instantly")

# round-trip check
rt = np.fromfile(track_bin, np.int16).reshape(-1, 4)
assert rt.shape[0] == n, f"binary has {rt.shape[0]} frames, expected {n}"
assert np.array_equal(rt, arr), "binary round-trip mismatch"
print("   round-trip verified")

# =============================================================================
# 2 · Thumbnail — from inside the longest rally
# =============================================================================
print("\n2 · THUMBNAIL")
lens = {}
for s in shots:
    lens[s["rally_id"]] = lens.get(s["rally_id"], 0) + 1
best_rally = max(lens, key=lens.get)
frames = sorted(s["frame"] for s in shots if s["rally_id"] == best_rally)
mid = frames[len(frames)//2]

cap = cv2.VideoCapture(str(mp4))          # the rendered video, so the skeleton shows
cap.set(cv2.CAP_PROP_POS_FRAMES, int(mid))
ok, img = cap.read()
cap.release()
if not ok:                                 # fall back to the source
    cap = cv2.VideoCapture(str(SRC))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(mid + bundle["clip"]["start_frame"]))
    ok, img = cap.read(); cap.release()
assert ok, "could not read a frame for the thumbnail"

h, w = img.shape[:2]
thumb = cv2.resize(img, (640, int(round(640*h/w/2)*2)), interpolation=cv2.INTER_AREA)
cv2.imwrite(str(thumb_p), thumb, [cv2.IMWRITE_JPEG_QUALITY, 82])
print(f"   {thumb_p.name:<22} {thumb_p.stat().st_size/1e3:7.0f} KB  "
      f"{thumb.shape[1]}x{thumb.shape[0]}")
print(f"   from rally {best_rally} ({lens[best_rally]} shots), frame {mid}")

# =============================================================================
# 3 · Validation
# =============================================================================
print("\n3 · VALIDATION")
fail = []

def check(cond, msg):
    print(f"   {'OK  ' if cond else 'FAIL'} {msg}")
    if not cond: fail.append(msg)

# --- the field the velocity gate depends on ---
fps = bundle.get("source_fps")
check(fps is not None, "source_fps present")
check(isinstance(fps, (int, float)) and fps > 0, f"source_fps valid ({fps})")
if fps:
    gate = "velocity metrics ENABLED" if fps >= 60 else "velocity metrics GATED OFF"
    print(f"        -> {fps:.0f}fps: {gate}")

# --- video vs track vs shots must agree ---
cap = cv2.VideoCapture(str(mp4))
vn  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); vfps = cap.get(cv2.CAP_PROP_FPS)
vw  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); vh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
check(vn == n, f"video frames {vn:,} == track frames {n:,}")
check(abs(vfps - header["fps"]) < 0.05, f"video fps {vfps:.2f} == track fps {header['fps']}")
check((vw, vh) == (header["video_w"], header["video_h"]),
      f"video {vw}x{vh} == track {header['video_w']}x{header['video_h']}")
check(abs(bundle["duration_s"] - vn/max(vfps,1)) < 0.1,
      f"duration_s {bundle['duration_s']:.2f} == video {vn/max(vfps,1):.2f}s")

# --- shots inside the video ---
ts = [s["timestamp_s"] for s in shots]
check(all(ts[i] <= ts[i+1] for i in range(len(ts)-1)), "timestamps monotonic")
check(max(ts) <= bundle["duration_s"] + 0.05,
      f"last shot {max(ts):.2f}s within duration {bundle['duration_s']:.2f}s")

# --- crop rects stay inside the frame, or drawImage samples out of bounds ---
sc = header["scale"]
for side, k in (("left", 0), ("right", 1)):
    cw = header["players"][side]["crop_w"]; ch = header["players"][side]["crop_h"]
    cx = rt[:, k*2].astype(np.float64)/sc
    cy = rt[:, k*2+1].astype(np.float64)/sc
    inside = ((cx - cw/2) >= -0.6).all() and ((cx + cw/2) <= header["video_w"]+0.6).all() \
         and ((cy - ch/2) >= -0.6).all() and ((cy + ch/2) <= header["video_h"]+0.6).all()
    check(inside, f"{side} crop rect stays inside the frame")
    zoom = 360/cw
    print(f"        {side}: crop {cw:.0f}x{ch:.0f}, {zoom:.2f}x in a 360px panel"
          + ("  (soft — raise OUT_W)" if zoom > 1.25 else "  (sharp)"))

# --- fields the UI reads ---
need = ["rally_id","shot_index","player","timestamp_s","shot_class",
        "class_confidence","abstain","backswing_amplitude","contact_height",
        "peak_wrist_speed","table_distance","rally_length"]
missing = [f for f in need if f not in shots[0]]
check(not missing, f"shot fields present{'' if not missing else f' — missing {missing}'}")

klass = {}
for s in shots: klass[s["shot_class"]] = klass.get(s["shot_class"], 0) + 1
ab = sum(1 for s in shots if s["abstain"])
check(set(klass) <= {"serve","attack","control","defence"}, f"shot classes valid {klass}")
check(0 < ab < len(shots), f"abstain flags present ({ab} of {len(shots)})")

# =============================================================================
print("\n" + "="*66)
print("BUNDLE  ->  copy to public/demo/")
print("="*66)
tot = 0
for p in (mp4, bundle_p, track_bin, track_p, thumb_p):
    mb = p.stat().st_size/1e6; tot += mb
    print(f"  {p.name:<24} {mb:8.1f} MB")
print(f"  {'TOTAL':<24} {tot:8.1f} MB")
if tot > 150:
    print(f"""
  Over 150 MB — heavy to commit. Either set CLIP_START_S / CLIP_END_S and
  re-export (the track and shots are cut and rebased automatically), or host
  the mp4 on R2/S3 and point bundle.video_url at it.""")

print(f"""
FRONTEND

  const hdr = await fetch('/demo/{VIDEO_ID}.track.json').then(r => r.json());
  const buf = await fetch('/demo/{VIDEO_ID}.track.bin').then(r => r.arrayBuffer());
  const t   = new Int16Array(buf);            // a view, not a parse

  const i  = Math.min(Math.round(video.currentTime * hdr.fps), hdr.n_frames - 1);
  const P  = hdr.players.left;
  const cx = t[i*4]     / hdr.scale;
  const cy = t[i*4 + 1] / hdr.scale;          // right player: i*4+2, i*4+3

  ctx.drawImage(video,
    cx - P.crop_w/2, cy - P.crop_h/2, P.crop_w, P.crop_h,
    0, 0, panelW, panelH);

NOT NEEDED for this demo
  - pose_track / pose_windows: the skeleton is baked into the video, so the
    frontend draws no keypoints at all
  - per-player rendered videos: the crop track replaces them

NOTES
  - centres are already smoothed and clamped — do not smooth again
  - they are in OUTPUT pixel space, so no rescaling
  - guard drawImage on video.readyState >= 2""")

print("\n" + ("ALL CHECKS PASSED" if not fail
               else f"{len(fail)} CHECK(S) FAILED:\n  - " + "\n  - ".join(fail)))

1 · CROP TRACK
   game_1.track.bin           709 KB  (int16 x88599 frames)
   game_1.track.json          0.4 KB  (header only)
   was 1.86 MB of JSON  ->  0.71 MB, parsed instantly
   round-trip verified

2 · THUMBNAIL
   game_1.thumb.jpg            36 KB  640x360
   from rally 23 (22 shots), frame 71837

3 · VALIDATION
   OK   source_fps present
   OK   source_fps valid (120.0)
        -> 120fps: velocity metrics ENABLED
   OK   video frames 88,599 == track frames 88,599
   OK   video fps 120.00 == track fps 120.0
   OK   video 1600x900 == track 1600x900
   OK   duration_s 738.33 == video 738.33s
   OK   timestamps monotonic
   OK   last shot 716.24s within duration 738.33s
   OK   left crop rect stays inside the frame
        left: crop 600x900, 0.60x in a 360px panel  (sharp)
   OK   right crop rect stays inside the frame
        right: crop 600x900, 0.60x in a 360px panel  (sharp)
   OK   shot fields present
   OK   shot classes valid {'defence': 61, 'serve': 26, 'attack': 63, 'con